# carga los puntos de recorridos de surcado a base de datos

In [1]:
import sys
sys.path.append('..')
from sqlalchemy import create_engine, text
import os

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

POSTGRES_UTEA['DATABASE'] = 'utea_precision'

In [2]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

def insert_datos_lote_con_verificacion(lista_nombres):
    engine = obtener_engine()
    
    for nombre_completo in lista_nombres:
        partes = nombre_completo.split('_')
        
        if len(partes) >= 4:
            u01 = int(partes[0])
            u02 = partes[1]
            u05 = partes[2]
            camp = int(partes[3])
            
            # 1. Query de verificación
            query_check = text("""
                SELECT 1 FROM siembra_surcado.data_lote 
                WHERE unidad_01 = :u01 AND unidad_05 = :u05
            """)
            
            # 2. Query de inserción
            query_insert = text("""
                INSERT INTO siembra_surcado.data_lote 
                (unidad_01, unidad_02, unidad_05, campanha)
                VALUES (:u01, :u02, :u05, :camp)
            """)
            
            try:
                with engine.begin() as conn:
                    # Ejecutar verificación
                    existe = conn.execute(query_check, {'u01': u01, 'u05': u05}).fetchone()
                    
                    if not existe:
                        conn.execute(query_insert, {
                            'u01': u01, 
                            'u02': u02, 
                            'u05': u05, 
                            'camp': camp
                        })
                        print(f"✅ Insertado: {u01} - {u05}")
                    else:
                        print(f"⚠️ Omitido (ya existe): {u01} - {u05}")
                        
            except Exception as e:
                print(f"❌ Error en {nombre_completo}: {e}")


In [3]:
# ruta de shps
PATH_SHPS = RUTA_UNIDAD_ONE_DRIVE + r"/Ingenio Azucarero Guabira S.A/UTEA - SEMANAL - SIEMBRA/SIEMBRA CON PRESICION/2026/SHP_PUNTOS"

In [4]:
# lista de nombres de archivos en ruta
nombres_shp = [
    os.path.splitext(archivo)[0] # separa nombre de archivo y extencion
    for archivo in os.listdir(PATH_SHPS) # recorre la lista
    if archivo.endswith('.shp') # solo los que terminan en .shp
]
# Ver el resultado
nombres_shp

['2238_EL PARAISO_L5-1_2026',
 '2238_EL PARAISO_L5-2_2026',
 '2238_EL PARAISO_L5_2026',
 '2238_EL PARAISO_L7.1_2026',
 '2238_EL PARAISO_L7.2_2026',
 '2250_COSORIOCITO_CS-6B_2026',
 '2250_COSORIOCITO_CS-8B_2026',
 '2250_COSORIOCITO_CS-9B_2026',
 '2250_COSORIOCITO_L10.1_2026',
 '2250_COSORIOCITO_L10.2_2026',
 '2250_COSORIOCITO_L11_2026',
 '2250_COSORIOCITO_L12_2026',
 '2250_COSORIOCITO_L7_2026',
 '2308_CAMBERRA--AGUILERA_C13_2026',
 '259_SANTA ANA DE PAILON_L18_2026',
 '30_CAMPO DULCE_EP-L16A_2026',
 '30_CAMPO DULCE_EP-L17_2026',
 '30_CAMPO DULCE_EP-L20-EP-L21_2026',
 '30_CAMPO DULCE_EP-L33-EP-L34_2026',
 '30_CAMPO DULCE_EP-L9_2026',
 '30_CAMPO DULCE_ER-L1.2_2026',
 '30_CAMPO DULCE_ER-L5-ER-L6_2026',
 '482_TEXAS_TX-1_2026']

In [5]:
# Ejecución
insert_datos_lote_con_verificacion(nombres_shp)

⚠️ Omitido (ya existe): 2238 - L5-1
⚠️ Omitido (ya existe): 2238 - L5-2
⚠️ Omitido (ya existe): 2238 - L5
⚠️ Omitido (ya existe): 2238 - L7.1
⚠️ Omitido (ya existe): 2238 - L7.2
✅ Insertado: 2250 - CS-6B
✅ Insertado: 2250 - CS-8B
✅ Insertado: 2250 - CS-9B
⚠️ Omitido (ya existe): 2250 - L10.1
⚠️ Omitido (ya existe): 2250 - L10.2
⚠️ Omitido (ya existe): 2250 - L11
⚠️ Omitido (ya existe): 2250 - L12
⚠️ Omitido (ya existe): 2250 - L7
⚠️ Omitido (ya existe): 2308 - C13
⚠️ Omitido (ya existe): 259 - L18
⚠️ Omitido (ya existe): 30 - EP-L16A
⚠️ Omitido (ya existe): 30 - EP-L17
⚠️ Omitido (ya existe): 30 - EP-L20-EP-L21
⚠️ Omitido (ya existe): 30 - EP-L33-EP-L34
⚠️ Omitido (ya existe): 30 - EP-L9
⚠️ Omitido (ya existe): 30 - ER-L1.2
⚠️ Omitido (ya existe): 30 - ER-L5-ER-L6
⚠️ Omitido (ya existe): 482 - TX-1
